# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [3]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'cuaca_indonesia_2015_2025.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: cuaca_indonesia_2015_2025.parquet
Ukuran file: 17.7 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [4]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'time': String, 'temperature_2m': Float64, 'relative_humidity_2m': Int64, 'precipitation': Float64, 'wind_speed_10m': Float64, 'kota': String, 'lat': Float64, 'lon': Float64})


jumlah_baris,time_null,temperature_2m_null,relative_humidity_2m_null,precipitation_null,wind_speed_10m_null,kota_null,lat_null,lon_null
u32,u32,u32,u32,u32,u32,u32,u32,u32
3278688,0,0,0,0,0,0,0,0


In [5]:
contoh_data = lazy_data.head(10).collect()
contoh_data

time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,kota,lat,lon
str,f64,i64,f64,f64,str,f64,f64
"""2015-01-01T00:00""",27.3,82,0.0,16.5,"""Ambon""",-3.7,128.18
"""2015-01-01T01:00""",27.3,82,0.0,17.4,"""Ambon""",-3.7,128.18
"""2015-01-01T02:00""",27.2,81,0.2,16.6,"""Ambon""",-3.7,128.18
"""2015-01-01T03:00""",27.3,80,0.0,17.0,"""Ambon""",-3.7,128.18
"""2015-01-01T04:00""",27.4,80,0.0,19.4,"""Ambon""",-3.7,128.18
"""2015-01-01T05:00""",27.4,80,0.0,22.9,"""Ambon""",-3.7,128.18
"""2015-01-01T06:00""",27.4,81,0.0,22.2,"""Ambon""",-3.7,128.18
"""2015-01-01T07:00""",27.3,81,0.0,21.9,"""Ambon""",-3.7,128.18
"""2015-01-01T08:00""",27.5,81,0.0,20.6,"""Ambon""",-3.7,128.18


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [6]:
connection = duckdb.connect()
profil_duckdb = connection.execute(
    'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
).pl()
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""time""","""VARCHAR""","""2015-01-01T00:00""","""2025-12-31T23:00""",112872,null,null,null,null,null,3278688,0.00
"""temperature_2m""","""DOUBLE""","""13.9""","""43.0""",243,"""26.43415631496519""","""2.3527682301919524""","""24.777363097030474""","""26.084054471133484""","""27.980896062063195""",3278688,0.00
"""relative_humidity_2m""","""BIGINT""","""13""","""100""",82,"""84.95728535316566""","""11.953279380941943""","""78""","""88""","""94""",3278688,0.00
"""precipitation""","""DOUBLE""","""0.0""","""61.6""",394,"""0.3282979350278965""","""1.0891382858226173""","""0.0""","""0.0""","""0.1939053258210601""",3278688,0.00
"""wind_speed_10m""","""DOUBLE""","""0.0""","""58.6""",392,"""6.815683590509757""","""4.440079262285683""","""3.661964210732596""","""5.6890573657911""","""8.847514752773128""",3278688,0.00
"""kota""","""VARCHAR""","""Ambon""","""Yogyakarta""",29,null,null,null,null,null,3278688,0.00
"""lat""","""DOUBLE""","""-10.18""","""5.55""",33,"""-2.801764705874747""","""3.7372868381420865""","""-6.12""","""-2.5948922937178884""","""-0.03""",3278688,0.00
"""lon""","""DOUBLE""","""95.32""","""140.72""",36,"""113.61147058839983""","""10.38592733554913""","""105.26""","""113.21928593455628""","""119.87""",3278688,0.00


## Catatan Milestone 1

### Ringkasan Dataset
- File: `cuaca_indonesia_2015_2025.parquet`, 17,7 MB (Parquet terkompresi).
- Jumlah baris: 3.278.688 (> 1 juta), 8 kolom.
- Cakupan: 34 ibu kota provinsi × 96.432 jam (1 Januari 2015 - 31 Desember 2025).

### Tipe Kolom
- `time`: String (format ISO `2015-01-01T00:00`), perlu dikonversi ke Datetime.
- `temperature_2m`, `precipitation`, `wind_speed_10m`, `lat`, `lon`: Float64.
- `relative_humidity_2m`: Int64. `kota`: String.

### Kualitas Data Awal
- Tidak ada null di semua kolom (null_percentage 0,00%), jadi tidak ada kolom dengan null terbanyak.
- Rentang nilai: suhu 13,9-43,0 °C (rata-rata 26,4); kelembapan 13-100% (rata-rata 85,0); curah hujan 0-61,6 mm/jam (rata-rata 0,33, median 0,0); kecepatan angin 0-58,6 (rata-rata 6,8).
- Kelembapan berada di rentang valid 0-100 dan tidak ada nilai negatif pada curah hujan maupun angin.

### Anomali untuk Milestone 2
1. Kolom `time` masih String, perlu di-cast ke Datetime sebelum analisis temporal.
2. Curah hujan sangat miring: median 0,0 dan kuartil ketiga hanya sekitar 0,19, tetapi maksimum 61,6 mm/jam. Perlu dicek apakah ini kejadian hujan ekstrem yang nyata atau outlier.
3. Suhu maksimum 43,0 °C tergolong tinggi untuk kota di Indonesia, dan kelembapan minimum 13% sangat rendah untuk iklim tropis. Perlu ditelusuri di kota dan waktu mana nilai itu muncul.
4. Nilai `approx_unique` dari DuckDB hanya perkiraan (misalnya `kota` terbaca 29 padahal seharusnya 34); jumlah unik yang tepat perlu dihitung dengan `n_unique`.
5. Pengecekan duplikat (kota + time) belum dilakukan.
6. `lat` dan `lon` konstan per kota (atribut lokasi), bukan hasil pengukuran, sehingga statistiknya tidak bermakna.